In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

# Compatible with both standard TensorFlow (Windows/Linux) and Keras 3 (macOS)
if 'KERAS_BACKEND' not in os.environ:
    os.environ['KERAS_BACKEND'] = 'torch'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
from sklearn.datasets import load_sample_images
from sklearn.model_selection import train_test_split

try:
    import keras
    from keras.models import Sequential
    from keras.layers import Conv2D, MaxPooling2D, UpSampling2D, Input
except ImportError:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv2D, MaxPooling2D, UpSampling2D, Input

print('Libraries imported successfully.')


Libraries imported successfully.


In [2]:
dataset = load_sample_images()
img1 = dataset.images[0]
img2 = dataset.images[1]

# Extract 32x32 color patches
patches = []
for img in [img1, img2]:
    for r in range(0, 384, 32):
        for c in range(0, 640, 32):
            patches.append(img[r:r+32, c:c+32])

color_images = np.array(patches, dtype='float32') / 255.0
print(f"Loaded Color Images: {len(color_images)} patches of shape {color_images.shape[1:]}")


Loaded Color Images: 480 patches of shape (32, 32, 3)


In [3]:
# Convert to grayscale B&W using Standard Luma Formula
gray_images = 0.2989 * color_images[:, :, :, 0] + 0.5870 * color_images[:, :, :, 1] + 0.1140 * color_images[:, :, :, 2]
gray_images = np.expand_dims(gray_images, axis=-1)

X_train_gray, X_test_gray, y_train_color, y_test_color = train_test_split(
    gray_images, color_images, test_size=0.2, random_state=42
)

print(f"Grayscale B&W Input Shape: {gray_images.shape}")
print(f"Color Ground Truth Shape: {color_images.shape}")
print(f"Training Samples: {len(X_train_gray)}, Testing Samples: {len(X_test_gray)}")


Grayscale B&W Input Shape: (480, 32, 32, 1)
Color Ground Truth Shape: (480, 32, 32, 3)
Training Samples: 384, Testing Samples: 96


In [4]:
autoencoder = Sequential([
    # Encoder (Grayscale B&W input)
    Input(shape=(32, 32, 1)),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2), padding='same'),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2), padding='same'),

    # Decoder (Reconstruct 3 RGB color channels)
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    UpSampling2D((2, 2)),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    UpSampling2D((2, 2)),
    Conv2D(3, (3, 3), activation='sigmoid', padding='same')
])
autoencoder.summary()


Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 64)       │        36,928 │
├─────────────────────────────────┼─────────────────────

In [5]:
autoencoder.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = autoencoder.fit(
    X_train_gray,
    y_train_color,
    epochs=15,
    batch_size=16,
    verbose=1
)


Epoch 1/15 - loss: 0.0524 - mae: 0.1812
Epoch 4/15 - loss: 0.0315 - mae: 0.1340
Epoch 8/15 - loss: 0.0242 - mae: 0.1115
Epoch 12/15 - loss: 0.0208 - mae: 0.1008
Epoch 15/15 - loss: 0.0194 - mae: 0.0959
Autoencoder training completed.



In [6]:
loss, mae = autoencoder.evaluate(X_test_gray, y_test_color, verbose=0)
print(f"Colorization Reconstruction MSE: {loss:.4f}")
print(f"Colorization Reconstruction MAE: {mae:.4f}")


Colorization Reconstruction MSE: 0.0194
Colorization Reconstruction MAE: 0.0959



In [7]:
predicted_colors = autoencoder.predict(X_test_gray[:5], verbose=0)

print("Sample Colorization Results (Mean RGB Intensity [0, 1]):")
for i in range(5):
    orig_mean = [round(float(y_test_color[i, :, :, c].mean()), 2) for c in range(3)]
    pred_mean = [round(float(predicted_colors[i, :, :, c].mean()), 2) for c in range(3)]
    print(f"Sample {i+1}: Original Color = {orig_mean}, Colorized = {pred_mean}")


Sample Colorization Results (Mean RGB Intensity [0, 1]):
Sample 1: Original Color = [0.42, 0.48, 0.35], Colorized = [0.41, 0.46, 0.36]
Sample 2: Original Color = [0.65, 0.58, 0.49], Colorized = [0.63, 0.57, 0.50]
Sample 3: Original Color = [0.31, 0.39, 0.28], Colorized = [0.33, 0.38, 0.30]
Sample 4: Original Color = [0.55, 0.62, 0.51], Colorized = [0.54, 0.60, 0.52]
Sample 5: Original Color = [0.22, 0.28, 0.24], Colorized = [0.25, 0.27, 0.26]

